# Notebook 01 - Local Benchmark Construction

This notebook constructs a reproducible local benchmark for Local Causal Propagation Analysis Task.

Required files:
- `adj_matrix_67.npy`
- `adj_matrix_67_binary.npy`
- `node_order_67.npy`
- `final_incidents_with_weather.csv`
- `final_hourly_flow_allfeature_with_timefeat.csv`

The benchmark combines

- road graph information
- incident records
- traffic observations

to generate a standardized benchmark package surrounding one selected incident.

Outputs include

- local traffic flow matrix
- incident metadata
- sensor metadata
- local road graph

These outputs are consumed by Notebook 02 for propagation analysis.

## Benchmark Construction Protocol

1. Raw benchmark datasets
2. Data consistency checking
3. Incident filtering
4. Representative incident selection
5. Local subgraph extraction
6. Local traffic extraction
7. Benchmark package export

## 1. Import libraries and define paths
Load all required libraries and define the benchmark directory structure.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("data/nsw2025")

## 2. Load Benchmark Datasets

The benchmark uses: road graph adjacency matrices; station metadata; incident-weather dataset; hourly traffic flow; generated during the benchmark preprocessing stage.

In [ ]:
adj_67 = np.load("adj_matrix_67.npy")
adj_67_binary = np.load("adj_matrix_67_binary.npy")
node_order_67 = np.load("node_order_67.npy", allow_pickle=True).astype(str)

incidents_weather = pd.read_csv(
    "final_incidents_with_weather.csv",
    dtype={"station_id": str, "incident_id": str},
    low_memory=False
)

flow_all = pd.read_csv(
    "final_hourly_flow_allfeature_with_timefeat.csv",
    dtype={"station_id": str, "incident_id": str},
    low_memory=False
)

## 3. Inspect dataset dimensions and columns

This quality-control step verifies that all benchmark inputs are successfully loaded before further processing. The following properties are inspected:

- dataset dimensions
- graph size
- station counts
- available variables

In [7]:
print("GRAPH FILES")
print("adj_67:", adj_67.shape)
print("adj_67_binary:", adj_67_binary.shape)
print("node_order_67:", node_order_67.shape)

print("\nTABLE FILES")
print("incidents_weather:", incidents_weather.shape)
print("flow_all:", flow_all.shape)

print("\nFLOW COLUMNS")
print(flow_all.columns.tolist())

print("\nINCIDENT-WEATHER COLUMNS")
print(incidents_weather.columns.tolist())

GRAPH FILES
adj_67: (67, 67)
adj_67_binary: (67, 67)
node_order_67: (67,)

TABLE FILES
incidents_weather: (3197, 26)
flow_all: (1007400, 31)

FLOW COLUMNS
['station_id', 'timestamp', 'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'is_holiday', 'hour', 'total_flow', 'wgs84_latitude', 'wgs84_longitude', 'road_name', 'suburb', 'post_code', 'device_type', 'quality_rating', 'lane_count', 'road_functional_hierarchy', 'distance_to_intersection', 'incident_id', 'incident_type', 'is_major_incident', 'impact_sequence_hour', 'precipitation', 'weather_code', 'apparent_temperature', 'temperature_2m', 'wind_gusts_10m', 'relative_humidity', 'incident_count', 'is_anomaly']

INCIDENT-WEATHER COLUMNS
['station_id', 'Abs PM_x', 'Abs PM_y', 'incident_id', 'dis', 'hazard_type', 'incident_kind', 'match_hour', 'calculated_duration_hours', 'impact_sequence_hour', 'is_major_incident', 'is_local_road', 'speed_limit', 'advice_a', 'advice_b', 'queue_length_km', 'traffic_volume_desc', 'affected_direction', '

## 4. Standardize identifiers and timestamps

Station identifiers and timestamps are standardized before integration to prevent mismatched joins.

In [8]:
# Standardize station IDs
flow_all["station_id"] = flow_all["station_id"].astype(str).str.strip()
incidents_weather["station_id"] = incidents_weather["station_id"].astype(str).str.strip()
node_order_67 = node_order_67.astype(str)

# Parse timestamps
flow_all["timestamp"] = pd.to_datetime(
    flow_all["timestamp"],
    format="mixed",
    errors="coerce"
)

incidents_weather["match_hour"] = pd.to_datetime(
    incidents_weather["match_hour"],
    format="mixed",
    errors="coerce"
)

# check missing value
print("Missing flow timestamps:", flow_all["timestamp"].isna().sum())
print("Missing incident match hours:", incidents_weather["match_hour"].isna().sum())

Missing flow timestamps: 0
Missing incident match hours: 0


## 5. Cross-Dataset Consistency Validation

Benchmark reproducibility requires every selected station to exist in traffic data, graph, incidents, and centrality rankings. This section verifies consistency across all benchmark components.

In [9]:
flow_ids = set(flow_all["station_id"])
incident_ids = set(incidents_weather["station_id"])
node67_ids = set(node_order_67)

print("Flow stations:", len(flow_ids))
print("Incident-weather stations:", len(incident_ids))
print("Node order 67 stations:", len(node67_ids))

print("\nFlow == node_order_67:", flow_ids == node67_ids)
print("Incidents == node_order_67:", incident_ids == node67_ids)

print("\nFlow not in node_order_67:", sorted(flow_ids - node67_ids))
print("Node_order_67 not in flow:", sorted(node67_ids - flow_ids))

Flow stations: 115
Incident-weather stations: 67
Node order 67 stations: 67

Flow == node_order_67: False
Incidents == node_order_67: True

Flow not in node_order_67: ['6105', '6107', '6109', '6110', '6113', '6114', '6119-PR', '6120-PR', '6124', '6126', '6145', '6146', '6147', '6148', '6149', '6152', '6167', '6168', '6194', '6196', '6197', '7212', '7216', '7217', '7220', '7289', 'HEXBUCHW-PR', 'HHW005', 'HHW006', 'MOHVCS', 'MUB001', 'NNGSTC', 'PHSTC', 'T0065', 'T0242', 'T0274-PR', 'T0284', 'T0294', 'T0295', 'T0296', 'T0297', 'T0298', 'T0299', 'T0384', 'T0482', 'T0485', 'T0494', 'T0496']
Node_order_67 not in flow: []


## 6. Validate Local Road Graph

The road graph defines spatial connectivity used throughout Task 3.

Both weighted and binary adjacency matrices are inspected to ensure the graph is complete and suitable for extracting local neighbourhoods.

In [10]:
print("WEIGHTED GRAPH")
print("Shape:", adj_67.shape)
print("Minimum:", adj_67.min())
print("Maximum:", adj_67.max())
print("Non-zero entries:", np.count_nonzero(adj_67))

print("\nBINARY GRAPH")
print("Shape:", adj_67_binary.shape)
print("Unique values:", np.unique(adj_67_binary))
print("Non-zero entries:", np.count_nonzero(adj_67_binary))

degree = adj_67_binary.sum(axis=1)

print("\nDEGREE SUMMARY")
print("Minimum degree:", degree.min())
print("Maximum degree:", degree.max())
print("Mean degree:", degree.mean())
print("Median degree:", np.median(degree))
print("Isolated stations:", np.sum(degree == 0))

WEIGHTED GRAPH
Shape: (67, 67)
Minimum: 0.0
Maximum: 36.96825043288063
Non-zero entries: 314

BINARY GRAPH
Shape: (67, 67)
Unique values: [0. 1.]
Non-zero entries: 314

DEGREE SUMMARY
Minimum degree: 2.0
Maximum degree: 9.0
Mean degree: 4.686567164179104
Median degree: 4.0
Isolated stations: 0


## 7. Inspect Candidate Incidents

The benchmark constructs one local benchmark around a representative incident. This section summarizes all available incidents before selection.

In [11]:
print("INCIDENT DATASET")

print("Rows:", len(incidents_weather))
print("Unique incidents:", incidents_weather["incident_id"].nunique())
print("Unique stations:", incidents_weather["station_id"].nunique())

print("\nHazard types")
print(incidents_weather["hazard_type"].value_counts())

print("\nIncident kinds")
print(incidents_weather["incident_kind"].value_counts())

print("\nMajor incidents")
print(incidents_weather["is_major_incident"].value_counts())

print("\nRoad type")
print(incidents_weather["is_local_road"].value_counts(dropna=False))

INCIDENT DATASET
Rows: 3197
Unique incidents: 1628
Unique stations: 67

Hazard types
hazard_type
BREAKDOWN                         1136
CRASH                              688
SPECIAL EVENT                      576
SCHEDULED ROADWORK                 145
HAZARD                             144
TRAFFIC LIGHTS BLACKED OUT         138
EMERGENCY ROADWORK                 103
CHANGED TRAFFIC CONDITIONS          86
ADVERSE WEATHER                     73
TRAFFIC LIGHTS FLASHING YELLOW      69
BUILDING FIRE                       16
FLOODING                            10
BUSHFIRE                             4
BURST WATER MAIN                     4
HEAVY TRAFFIC                        2
LATE FINISHING ROADWORK              1
HOLIDAY TRAFFIC                      1
GRASS FIRE                           1
Name: count, dtype: int64

Incident kinds
incident_kind
Unplanned    2445
Planned       752
Name: count, dtype: int64

Major incidents
is_major_incident
0    2344
1     853
Name: count, dtype: int64

R

## 8. Incident Duration Summary

Incident duration provides an indication of disruption severity and is later considered when selecting representative benchmark cases.

In [12]:
print(incidents_weather[
    [
        "calculated_duration_hours",
        "impact_sequence_hour",
        "queue_length_km",
        "precipitation"
    ]
    ].describe())

       calculated_duration_hours  impact_sequence_hour  queue_length_km  \
count                3197.000000           3197.000000           3197.0   
mean                   30.704125              3.830466              0.0   
std                    50.446992              8.242494              0.0   
min                     0.500000              0.000000              0.0   
25%                     0.500000              0.000000              0.0   
50%                     1.190204              0.000000              0.0   
75%                    75.208074              2.000000              0.0   
max                   120.000000             71.000000              0.0   

       precipitation  
count    3197.000000  
mean        0.140444  
std         0.627723  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max        13.500000  


## 9. Explore Crash Incidents

Task 3 focuses on **crashes** because they generate localized traffic disturbances that propagate through nearby roads.

In [13]:
crash = incidents_weather[incidents_weather["hazard_type"] == "CRASH"].copy()

print("Rows:", len(crash))
print("Unique crash incidents:", crash["incident_id"].nunique())
print("Stations with crashes:", crash["station_id"].nunique())

print("\nCrash duration summary")

print(crash[
    [
        "calculated_duration_hours",
        "impact_sequence_hour",
        "dis"
    ]
    ].describe())

Rows: 688
Unique crash incidents: 513
Stations with crashes: 61

Crash duration summary
       calculated_duration_hours  impact_sequence_hour         dis
count                 688.000000            688.000000  688.000000
mean                    1.007167              0.251453    0.308205
std                     0.801681              0.582994    0.228021
min                     0.500000              0.000000    0.003000
25%                     0.500000              0.000000    0.104000
50%                     0.613842              0.000000    0.272000
75%                     1.161134              0.000000    0.478500
max                     3.797086              3.000000    0.972000


## 10. Select Representative Local Incident

The representative incident is selected according to the following criteria:

- the event is classified as a **CRASH**;
- the event identified as a **major incident** whenever available;
- among major crashes, incidents are ranked by **min distance** and **calculated duration**;
- the matched traffic station must belong to the 67-node causal benchmark and contain complete hourly traffic observations;

In [14]:
# CRASH type
crash = incidents_weather[incidents_weather["hazard_type"] == "CRASH"].copy()

print("Crash rows:", len(crash))
print("Unique crash incidents:", crash["incident_id"].nunique())
print("Stations with crashes:", crash["station_id"].nunique())

# MAJOR incident
major_crash = crash[crash["is_major_incident"] == 1].copy()

print("\nMajor crash rows:", len(major_crash))
print("Unique major crashes:", major_crash["incident_id"].nunique())
print("Stations with major crashes:", major_crash["station_id"].nunique())

# Summarise
major_crash_summary = (major_crash.groupby(["incident_id", "station_id"]).agg(
    min_distance=("dis", "min"),
    duration_hours=("calculated_duration_hours", "first"),
    max_impact_hour=("impact_sequence_hour", "max"),
    hourly_records=("impact_sequence_hour", "count"),
    precipitation=("precipitation", "mean")
    )
    .reset_index())

# Rank incident candidate
major_crash_summary = major_crash_summary.sort_values(
    ["duration_hours", "hourly_records", "min_distance"],
    ascending=[False, False, True])

candidate_incidents = major_crash_summary[
    (major_crash_summary["duration_hours"] >= 3.0) &
    (major_crash_summary["hourly_records"] >= 4)
].copy()

candidate_incidents = candidate_incidents.sort_values(
    ["min_distance", "duration_hours", "hourly_records"],
    ascending=[True, False, False]
)

candidate_incidents.head(10)

Crash rows: 688
Unique crash incidents: 513
Stations with crashes: 61

Major crash rows: 688
Unique major crashes: 513
Stations with major crashes: 61


,incident_id,station_id,min_distance,duration_hours,max_impact_hour,hourly_records,precipitation
256,240298-webtirf,7119-PR,0.004,3.304605,3,4,0.000
333,245772-webtirf,7159,0.030,3.395790,3,4,0.050
307,243960.0-webtirf,7179,0.038,3.021994,3,4,0.175
227,238403-webtirf,9849,0.064,3.007141,3,4,0.000
296,242908-webtirf,7121,0.086,3.671055,3,4,0.000
295,242908-webtirf,7120,0.120,3.671055,3,4,0.000
414,252572-webtirf,7179,0.353,3.614813,3,4,0.000
251,239814-webtirf,33014,0.381,3.135563,3,4,0.000
521,259131-webtirf,7121,0.603,3.428626,3,4,0.000
520,259131-webtirf,7120,0.631,3.428626,3,4,0.000


A single representative incident is selected to construct a reproducible benchmark case.

Selection considers incident severity, nearest sensor distance, duration, and data completeness. The benchmark currently selects one case, although the same protocol can generate benchmark packages for any incident.

In [15]:
# select first incident
selected_event = candidate_incidents.iloc[0]

selected_incident_id = selected_event["incident_id"]
selected_station = str(selected_event["station_id"])

print("Selected incident:", selected_incident_id)
print("Selected station:", selected_station)
selected_event

Selected incident: 240298-webtirf
Selected station: 7119-PR


incident_id        240298-webtirf
station_id                7119-PR
min_distance                0.004
duration_hours           3.304605
max_impact_hour                 3
hourly_records                  4
precipitation                 0.0
Name: 256, dtype: object

## 11. Extract Local Benchmark Graph

After selecting the representative incident, its corresponding traffic station is used as the centre of the local causal analysis. The local benchmark graph is constructed by extracting the station and its adjacent stations from the benchmark graph. Only this local network is analysed in Task 3 to improve interpretability while preserving relevant propagation pathways.

In [16]:
selected_station = str(selected_station)

# Index of selected station
selected_idx = np.where(node_order_67 == selected_station)[0][0]

# adjacent stations/neighbour
neighbor_idx = np.where(adj_67_binary[selected_idx] == 1)[0]
neighbor_ids = node_order_67[neighbor_idx]
selected_stations = np.concatenate(([selected_station], neighbor_ids))

print("Selected station:", selected_station)
print("Incident:", selected_incident_id)
print("\nNeighbour stations:")
print(neighbor_ids)
print("\nTotal stations:", len(selected_stations))

Selected station: 7119-PR
Incident: 240298-webtirf

Neighbour stations:
['50240' '68025' '7164' '7179']

Total stations: 5


In [17]:
local_adj = adj_67[np.ix_(
    np.concatenate(([selected_idx], neighbor_idx)),
    np.concatenate(([selected_idx], neighbor_idx))
)]

local_adj_binary = adj_67_binary[np.ix_(
    np.concatenate(([selected_idx], neighbor_idx)),
    np.concatenate(([selected_idx], neighbor_idx))
)]

print("Weighted graph:", local_adj.shape)
print("Binary graph:", local_adj_binary.shape)

print("Degree of selected station:",
      int(adj_67_binary[selected_idx].sum()))

Weighted graph: (5, 5)
Binary graph: (5, 5)
Degree of selected station: 4


## 12. Construct Local Benchmark Package

Traffic observations, incident records and station metadata are extracted for every station contained within the local subgraph. These datasets form the benchmark inputs used during propagation analysis.

In [18]:
station_meta = (
    flow_all[
        flow_all["station_id"].isin(selected_stations)
    ][
        [
            "station_id",
            "wgs84_latitude",
            "wgs84_longitude",
            "road_name",
            "suburb",
            "post_code",
            "device_type",
            "quality_rating",
            "lane_count",
            "road_functional_hierarchy",
            "distance_to_intersection",
        ]
    ]
    .drop_duplicates()
    .sort_values("station_id")
)

print(station_meta)

       station_id  wgs84_latitude  wgs84_longitude              road_name  \
78840       50240      -33.796436       150.987595            Briens Road   
341640      68025      -33.816696       150.976623  Great Western Highway   
359160    7119-PR      -33.817974       150.981827        Hawkesbury Road   
473040       7164      -33.815620       150.969238  Great Western Highway   
490560       7179      -33.817272       150.962784            Jersey Road   

                      suburb  post_code device_type  quality_rating  \
78840              Northmead       2152       Tirtl               4   
341640  South Wentworthville       2145       Tirtl               4   
359160            Merrylands       2160       Tirtl               5   
473040  South Wentworthville       2145       Tirtl               5   
490560  South Wentworthville       2145       Tirtl               5   

        lane_count  road_functional_hierarchy  distance_to_intersection  
78840            2                  

In [19]:
local_incidents = (
    incidents_weather[
        incidents_weather["station_id"].isin(selected_stations)
    ]
    .copy()
)

print("Rows:", len(local_incidents))
print("Unique incidents:", local_incidents["incident_id"].nunique())

local_incidents.head()

Rows: 366
Unique incidents: 281


,station_id,Abs PM_x,Abs PM_y,incident_id,dis,hazard_type,incident_kind,match_hour,calculated_duration_hours,impact_sequence_hour,...,traffic_volume_desc,affected_direction,temperature_2m,rain,precipitation,weather_code,apparent_temperature,relative_humidity,wind_gusts_10m,dew_point_2m
3,7179,-33.817272,150.962784,220424-webtirf,0.285,CRASH,Unplanned,2025-01-13 21:00:00,0.500000,0,...,NaN,Southbound,24.20,0.0,0.0,3.0,28.105839,84.389980,11.159999,21.40
7,7179,-33.817272,150.962784,222027-webtirf,0.041,BREAKDOWN,Unplanned,2025-01-28 01:00:00,1.152000,0,...,NaN,Southbound,23.40,0.0,0.0,1.0,27.933231,94.409520,9.000000,22.45
8,7179,-33.817272,150.962784,222027-webtirf,0.041,BREAKDOWN,Unplanned,2025-01-28 02:00:00,1.152000,1,...,NaN,Southbound,22.95,0.3,0.3,51.0,26.700270,93.246020,16.199999,21.80
12,7179,-33.817272,150.962784,220200.0-webtirf,0.144,BREAKDOWN,Unplanned,2025-01-10 21:00:00,0.504598,0,...,NaN,Westbound,19.90,0.3,0.3,51.0,22.657019,93.384995,10.440001,18.80
42,7119-PR,-33.817974,150.981827,220419-webtirf,0.007,TRAFFIC LIGHTS BLACKED OUT,Unplanned,2025-01-13 19:00:00,1.257248,0,...,NaN,All directions,26.50,0.0,0.0,2.0,29.635155,72.038770,24.119999,21.05


In [20]:
flow_local = (
    flow_all[
        flow_all["station_id"].isin(selected_stations)
    ][
        [
            "timestamp",
            "station_id",
            "total_flow",
        ]
    ]
    .copy()
)

# Check duplicate
dup_count = flow_local.duplicated(
    subset=["timestamp", "station_id"]
).sum()

print("Duplicate station-hour rows:", dup_count)

flow_matrix = (
    flow_local
    .pivot_table(
        index="timestamp",
        columns="station_id",
        values="total_flow",
        aggfunc="mean"
    )
    .sort_index()
)

# Match local graph order
flow_matrix = flow_matrix[selected_stations]

print("Flow matrix shape:", flow_matrix.shape)
print("Missing values:", flow_matrix.isna().sum().sum())
flow_matrix.head()

Duplicate station-hour rows: 0
Flow matrix shape: (8760, 5)
Missing values: 0


station_id,7119-PR,50240,68025,7164,7179
timestamp,,,,,
2025-01-01 00:00:00,481.0,1224.0,472.0,0.0,0.0
2025-01-01 01:00:00,323.0,1856.0,376.0,0.0,0.0
2025-01-01 02:00:00,265.0,1241.0,316.0,0.0,0.0
2025-01-01 03:00:00,170.0,605.0,218.0,0.0,0.0
2025-01-01 04:00:00,136.0,471.0,164.0,0.0,173.0


Coverage validation ensures every selected station contains complete hourly observations over the study period. Incomplete stations may bias historical propagation estimation.

In [21]:
coverage = (
    flow_local
    .groupby("station_id")
    .agg(
        min_time=("timestamp", "min"),
        max_time=("timestamp", "max"),
        n_records=("timestamp", "nunique")
    )
    .reindex(selected_stations)
)

coverage

,min_time,max_time,n_records
station_id,,,
7119-PR,2025-01-01,2025-12-31 23:00:00,8760
50240,2025-01-01,2025-12-31 23:00:00,8760
68025,2025-01-01,2025-12-31 23:00:00,8760
7164,2025-01-01,2025-12-31 23:00:00,8760
7179,2025-01-01,2025-12-31 23:00:00,8760


## 13. Save Local Benchmark Files


Export the processed local benchmark datasets. These files serve as the standardized inputs for the local causal propagation analysis and ensure the experiment can be reproduced consistently.

The exported files include:

- `sensor_meta_info_local.csv`: Metadata for the selected station and its one-hop neighbours.
- `incidents_local.csv`: Incident records associated with the local benchmark graph.
- `raw_flow_local.npy`: Hourly traffic flow matrix for the local benchmark graph.
- `raw_flow_local.csv`: Traffic flow matrix in tabular format for inspection.

In [22]:
LOCAL_DIR = ROOT / "data"
LOCAL_DIR.mkdir(exist_ok=True)

station_meta.to_csv(
    LOCAL_DIR / "sensor_meta_info_local.csv",
    index=False
)

local_incidents.to_csv(
    LOCAL_DIR / "incidents_local.csv",
    index=False
)

flow_matrix.to_csv(
    LOCAL_DIR / "raw_flow_local.csv"
)

np.save(
    LOCAL_DIR / "raw_flow_local.npy",
    flow_matrix.values.astype(np.float32)
)

In [23]:
#subgraph information

local_graph = pd.DataFrame({
    "station_id": selected_stations
})

local_graph["graph_index"] = range(len(local_graph))

local_graph.to_csv(
    LOCAL_DIR / "local_graph_info.csv",
    index=False
)

local_graph

,station_id,graph_index
0,7119-PR,0
1,50240,1
2,68025,2
3,7164,3
4,7179,4


## 14. Save Local Benchmark Summary

The benchmark summary records the selected incident together with key metadata required to reproduce the benchmark case. This metadata serves as the benchmark identifier for subsequent analyses.

In [24]:
summary = pd.DataFrame({
    "selected_incident": [selected_incident_id],
    "selected_station": [selected_station],
    "hazard_type": ["CRASH"],
    "major_incident": [1],
    "incident_duration_hours": [selected_event["duration_hours"]],
    "incident_distance_km": [selected_event["min_distance"]],
    "max_impact_hour": [selected_event["max_impact_hour"]],
    "hourly_records": [selected_event["hourly_records"]],
    "num_local_stations": [len(selected_stations)],
    "num_local_incidents": [local_incidents["incident_id"].nunique()],
    "num_flow_hours": [flow_matrix.shape[0]],
    "flow_variables": [flow_matrix.shape[1]]
})

summary

,selected_incident,selected_station,hazard_type,major_incident,incident_duration_hours,incident_distance_km,max_impact_hour,hourly_records,num_local_stations,num_local_incidents,num_flow_hours,flow_variables
0,240298-webtirf,7119-PR,CRASH,1,3.304605,0.004,3,4,5,281,8760,5


In [28]:
summary.to_csv(
    LOCAL_DIR / "local_case_summary.csv",
    index=False
)